# 1. Setup and Data Preparation
Prepare the runtime environment and load the preprocessed dataset for training.

## Mount Google Drive
Mount Google Drive so the notebook can access the shared dataset archive and any saved outputs.

In [3]:
import sys

# 若在 Colab 環境執行，才掛載 Google Drive
if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
else:
    print("非 Colab 環境 (本機執行)，跳過掛載 Google Drive。")

非 Colab 環境 (本機執行)，跳過掛載 Google Drive。


## Extract the Dataset Archive
Unzip the preprocessed dataset into the Colab runtime so file access is fast during training and evaluation.

In [4]:
import os
import sys

# 僅在 Colab 環境執行解壓縮
if 'google.colab' in sys.modules:
    # 將雲端硬碟的 .zip 檔解壓縮到 Colab 本地空間 (讀取速度最快)
    zip_path = "/content/drive/MyDrive/processed_hagrid_small.zip" 
    extract_path = "/content/processed_hagrid_small"

    if os.path.exists(zip_path) and not os.path.exists(extract_path):
        print("解壓縮資料集至 Colab 空間中！ (這會大幅提升訓練速度)...")
        !unzip -q "{zip_path}" -d /content/
        print("解壓縮完成！")
    elif os.path.exists(extract_path):
        print("資料集已解壓縮過了。")
    else:
        print(f"請確認您的雲端硬碟根目錄真的有 {zip_path} 這個檔案！")
else:
    print("本機環境，跳過解壓縮。請確保資料集已經放在正確的資料夾中。")

本機環境，跳過解壓縮。請確保資料集已經放在正確的資料夾中。


## Load the Manifest Table
Read the CSV manifest and normalize stored file paths so they point to the extracted Colab dataset location.

In [5]:
import pandas as pd
from pathlib import Path

# Colab 測試時的預設解壓縮路徑
DATA_CSV = Path("/content/processed_hagrid_small/labels.csv")

if not DATA_CSV.exists():
    # 本機測試時的備用路徑 (Windows / Mac) D:\Hand_Gesture\data\processed_hagrid_small_detect
    DATA_CSV = Path("processed_hagrid_small_detect/labels.csv")
    if not DATA_CSV.exists():
        # 如果還是找不到，可以讓隊友修改這個絕對路徑 (例如 Windows D 槽)
        DATA_CSV = Path("D:/Hand_Gesture/data/processed_hagrid_small_detect/labels.csv")

df = pd.read_csv(DATA_CSV)

# === 修復路徑：支援各平台 ===
# 無論 CSV 內存的是 Mac 還是 Windows 的絕對路徑
# 統一拿相對結構 "crops/檔名" 組合上目前 CSV 實際所在的資料夾
base_dir = DATA_CSV.parent

def fix_path_cross_platform(path_str):
    if pd.isna(path_str): return path_str
    p_path = Path(path_str)
    # 取出 "crops/檔名" 或 "landmarks/檔名"，與真正的 base_dir 結合
    return str(base_dir / p_path.parent.name / p_path.name)

df["crop_path"] = df["crop_path"].apply(fix_path_cross_platform)
df["landmark_path"] = df["landmark_path"].apply(fix_path_cross_platform)

print("using:", DATA_CSV)
print(df.head())
print("total:", len(df))
print(df["label_name"].value_counts())
print(df["label"].value_counts().sort_index())

using: D:\Hand_Gesture\data\processed_hagrid_small_detect\labels.csv
   idx                                          crop_path  \
0    0  D:\Hand_Gesture\data\processed_hagrid_small_de...   
1    1  D:\Hand_Gesture\data\processed_hagrid_small_de...   
2    2  D:\Hand_Gesture\data\processed_hagrid_small_de...   
3    3  D:\Hand_Gesture\data\processed_hagrid_small_de...   
4    4  D:\Hand_Gesture\data\processed_hagrid_small_de...   

                                       landmark_path  label label_name  \
0  D:\Hand_Gesture\data\processed_hagrid_small_de...      0        N_A   
1  D:\Hand_Gesture\data\processed_hagrid_small_de...      0        N_A   
2  D:\Hand_Gesture\data\processed_hagrid_small_de...      0        N_A   
3  D:\Hand_Gesture\data\processed_hagrid_small_de...      0        N_A   
4  D:\Hand_Gesture\data\processed_hagrid_small_de...      0        N_A   

   source_label source_label_name  
0             0              call  
1             0              call  
2          

## Split: Training / Validation / Testing
Create stratified train, validation, and test splits so each class keeps a similar distribution across all subsets.

In [6]:
from sklearn.model_selection import train_test_split

SEED = 42

# 如果需要測試用少量樣本，可以取消註解底下這行覆寫：
# df = pd.read_csv("D:/Hand_Gesture/data/processed_sample/labels.csv")

# Split train 70%, temp 30%
train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    random_state=SEED,
    stratify=df["label"]
)

# Split temp 30% into val 15% and test 15%
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=SEED,
    stratify=temp_df["label"]
)

# Reset index for all DataFrames
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("using:", DATA_CSV)
print("train:", len(train_df))
print("val:", len(val_df))
print("test:", len(test_df))

print("\nTrain class count:")
print(train_df["label_name"].value_counts())

print("\nVal class count:")
print(val_df["label_name"].value_counts())

print("\nTest class count:")
print(test_df["label_name"].value_counts())

using: D:\Hand_Gesture\data\processed_hagrid_small_detect\labels.csv
train: 98656
val: 21141
test: 21141

Train class count:
label_name
N_A     75800
palm     4814
ok       4642
one      4595
fist     4411
like     4394
Name: count, dtype: int64

Val class count:
label_name
N_A     16243
palm     1032
ok        994
one       985
fist      945
like      942
Name: count, dtype: int64

Test class count:
label_name
N_A     16243
palm     1031
ok        995
one       985
fist      946
like      941
Name: count, dtype: int64


# 2. Image Encoder Preparation
Set up the pretrained image backbone and verify its feature output before multimodal fusion.

In [3]:
import torch
import torch.nn as nn
from torchvision.models import mobilenet_v3_small, MobileNet_V3_Small_Weights

# 支援跨平台：NVIDIA CUDA (Windows/Linux), Apple MPS (Mac M1/M2), 或是只用 CPU
if torch.cuda.is_available():
    device = torch.device("cuda")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print("device:", device)

# load MobileNetV3-small with pretrained weights on ImageNet
weights = MobileNet_V3_Small_Weights.IMAGENET1K_V1
model = mobilenet_v3_small(weights=weights)

# MobileNetV3-small was originally trained on ImageNet with 1000 classes,
# but we need to adapt it for our 6 classes.
model.classifier[3] = nn.Linear(model.classifier[3].in_features, 6)

model = model.to(device)

device: cuda


## Check the Runtime Device
Print the installed PyTorch and CUDA information so you can confirm whether training will run on GPU or CPU.

In [4]:
import torch

print("torch version:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("mps available:", hasattr(torch.backends, "mps") and torch.backends.mps.is_available())
if torch.cuda.is_available():
    print("torch cuda version:", torch.version.cuda)
    print("device count:", torch.cuda.device_count())

torch version: 2.8.0+cu129
cuda available: True
mps available: False
torch cuda version: 12.9
device count: 1


## Define Image Transforms
Install the required libraries and build the image preprocessing pipeline expected by MobileNetV3-Small.

In [5]:
import sys
!{sys.executable} -m pip install torch torchvision
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from torchvision import transforms

# standard transformation for MobileNetV3-small, which expects 224x224 images and normalization with ImageNet's mean and std.
base_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

train_transform = base_transform
val_transform = base_transform
test_transform = base_transform

## Build the Feature Extractor
Build a compact image encoder and verify that each input crop is converted into the expected feature embedding.

In [6]:
import torch
import torch.nn as nn
from torchvision.models import mobilenet_v3_small, MobileNet_V3_Small_Weights


class MobileNetV3SmallFeatureExtractor(nn.Module):
    def __init__(self, output_dim=128):
        super().__init__()

        # Start from ImageNet-pretrained MobileNetV3-small.
        weights = MobileNet_V3_Small_Weights.IMAGENET1K_V1
        backbone = mobilenet_v3_small(weights=weights)

        # Keep the CNN backbone and global average pooling from MobileNetV3-small.
        self.features = backbone.features
        self.avgpool = backbone.avgpool

        # The pooled feature is typically 576-D, then projected to a compact embedding.
        self.projector = nn.Sequential(
            nn.Linear(576, output_dim),
            nn.ReLU(),
            nn.Dropout(0.2)
        )

    def forward(self, x):
        # Input: image tensor of shape [B, 3, H, W]
        # Output: feature vector of shape [B, output_dim]
        x = self.features(x) # -> [B, 576, h, w]
        x = self.avgpool(x) # -> [B, 576, 1, 1]
        x = torch.flatten(x, 1) # -> [B, 576]
        x = self.projector(x) # -> [B, output_dim]
        return x

## Instantiate the Image Encoder
Create a standalone image encoder module that maps each hand crop into a 128-dimensional representation.

In [7]:
# Build a standalone image encoder that maps each hand crop to a 128-D feature vector.
image_encoder = MobileNetV3SmallFeatureExtractor(output_dim=128)
image_encoder = image_encoder.to(device)

## Verify Encoder Output Shapes
Run a single mini-batch through the encoder to confirm the output dimensionality before using it in fusion.

In [8]:
# Create a dummy mini-batch and verify that the image encoder produces the expected feature size.
# Since train_loader is defined later in the fusion section, we use dummy data for now.
imgs = torch.randn(8, 3, 224, 224).to(device)

# Use evaluation mode because we only want a forward pass for inspection.
image_encoder.eval()

with torch.no_grad():
    features = image_encoder(imgs)

print("imgs shape:", imgs.shape)
print("features shape:", features.shape)

imgs shape: torch.Size([8, 3, 224, 224])
features shape: torch.Size([8, 128])


# 3. Fusion Model Training
Combine image and landmark features, then train the multimodal classifier.

In [9]:
import torch
import torch.nn as nn

class LandmarkMLP(nn.Module):
    def __init__(self, input_dim=42, hidden_dim=64, output_dim=128):
        super().__init__()

        # The landmark branch encodes 21 (x, y) points into a learned feature vector.
        self.mlp = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            # normalizing the activations can help stabilize training and improve generalization,
            # especially when the input features have varying scales or distributions.
            nn.BatchNorm1d(hidden_dim),
            # prevent overfitting by randomly zeroing out 20% of activations during training,
            # only applied to the first hidden layer since the model is already quite small.
            nn.Dropout(0.2),

            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.BatchNorm1d(hidden_dim),

            nn.Linear(hidden_dim, output_dim)
        )

    def forward(self, landmarks):
        # Landmarks may arrive as [B, 21, 2] or already flattened as [B, 42].
        if landmarks.dim() == 3:
            landmarks = landmarks.view(landmarks.size(0), -1)

        return self.mlp(landmarks)


class GestureFusionModel(nn.Module):
    def __init__(self, image_dim=128, landmark_dim=128, num_classes=6):
        super().__init__()

        # Encode image and landmark inputs separately.
        self.image_encoder = MobileNetV3SmallFeatureExtractor(output_dim=image_dim)
        self.landmark_encoder = LandmarkMLP(output_dim=landmark_dim)

        # Concatenate both embeddings before classification.
        fusion_dim = image_dim + landmark_dim

        self.classifier = nn.Sequential(
            nn.Linear(fusion_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )

    def forward(self, img, landmarks):
        # image_feature: [B, 128]
        image_feature = self.image_encoder(img)

        # landmark_feature: [B, 128]
        landmark_feature = self.landmark_encoder(landmarks)

        # fusion_feature: [B, 256]
        fusion_feature = torch.cat([image_feature, landmark_feature], dim=1)

        # logits: [B, num_classes]
        logits = self.classifier(fusion_feature)
        return logits

## Build the Fusion Dataset
Define a dataset that loads both the cropped image and the landmark array for each sample.

In [ ]:
import torch
from torch.utils.data import Dataset
from PIL import Image
import numpy as np

from augmentor_fixed import UltimateDataAugmentor
from padding_resize import pad_to_square_and_resize


class GestureFusionDataset(Dataset):
    def __init__(self, df, transform=None, is_train=True):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        self.is_train = is_train
        if self.is_train:
            self.augmentor = UltimateDataAugmentor(
                    p_blur=0.5, p_color=0.5, p_flip=0.5, 
                    max_rotate=15.0, max_scale=0.05, max_translate=0.03
                )

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        # Each row contains the saved crop image path, landmark path, and class label.
        row = self.df.iloc[idx]

        img = Image.open(row["crop_path"]).convert("RGB")
        landmarks = np.load(row["landmark_path"]).astype(np.float32)
        label = int(row["label"])

        # Padding and resize to square
        img, landmarks = pad_to_square_and_resize(img, landmarks, target_size=224)

        # Data Argumentation
        if self.is_train:
            img, landmarks = self.augmentor(img, landmarks)

        # Apply the same image preprocessing used by the image-only model.
        if self.transform is not None:
            img = self.transform(img)

        # Normalize landmarks by centering on the wrist and scaling by the maximum distance to any landmark.
        wrist_coords = landmarks[0, :]
        rel_landmarks = landmarks - wrist_coords
        max_dist = np.max(np.abs(rel_landmarks))
        if max_dist > 0:
            norm_landmarks = rel_landmarks / max_dist

        # Convert both modalities to tensors for PyTorch.
        landmarks = torch.tensor(norm_landmarks, dtype=torch.float32)
        label = torch.tensor(label, dtype=torch.long)

        return img, landmarks, label

## Create Fusion DataLoaders
Wrap the fusion datasets in DataLoaders so training and evaluation can iterate over mini-batches.

In [11]:
# Build train/validation/test datasets for the fusion model.
train_dataset = GestureFusionDataset(train_df, transform=train_transform, is_train=True)
val_dataset = GestureFusionDataset(val_df, transform=val_transform, is_train=False)
test_dataset = GestureFusionDataset(test_df, transform=val_transform, is_train=False)

BATCH_SIZE = 32

# DataLoader groups samples into mini-batches.
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)


## Initialize the Fusion Network
Instantiate the full multimodal model and move it onto the selected compute device.

In [12]:
# Create the full fusion model and move it to the selected device.
model = GestureFusionModel(
    image_dim=128,
    landmark_dim=128,
    num_classes=6
).to(device)

print(model)

GestureFusionModel(
  (image_encoder): MobileNetV3SmallFeatureExtractor(
    (features): Sequential(
      (0): Conv2dNormActivation(
        (0): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (1): BatchNorm2d(16, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
        (2): Hardswish()
      )
      (1): InvertedResidual(
        (block): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(16, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), groups=16, bias=False)
            (1): BatchNorm2d(16, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
            (2): ReLU(inplace=True)
          )
          (1): SqueezeExcitation(
            (avgpool): AdaptiveAvgPool2d(output_size=1)
            (fc1): Conv2d(16, 8, kernel_size=(1, 1), stride=(1, 1))
            (fc2): Conv2d(8, 16, kernel_size=(1, 1), stride=(1, 1))
            (activation): ReLU()
            (scale_activation): Hardsigmoid

## Sanity Check Fusion Shapes
Pass one batch through the fusion model to confirm that image tensors, landmark tensors, and logits all match the expected shapes.

In [13]:
# Run one batch through the fusion model to verify tensor shapes.
imgs, landmarks, labels = next(iter(train_loader))

imgs = imgs.to(device)
landmarks = landmarks.to(device)

with torch.no_grad():
    logits = model(imgs, landmarks)

print("imgs:", imgs.shape)
print("landmarks:", landmarks.shape)
print("labels:", labels.shape)
print("logits:", logits.shape)

imgs: torch.Size([32, 3, 224, 224])
landmarks: torch.Size([32, 21, 2])
labels: torch.Size([32])
logits: torch.Size([32, 6])


## Define Fusion Training Utilities
Implement the training and validation functions used to optimize the multimodal classifier.

In [14]:
def train_one_epoch(model, loader):
    model.train()

    total_loss = 0
    correct = 0
    total = 0

    for imgs, landmarks, labels in tqdm(loader, desc="Training", leave=False):
        imgs = imgs.to(device)
        landmarks = landmarks.to(device)
        labels = labels.to(device)

        # Clear old gradients before the next optimization step.
        optimizer.zero_grad()

        # Forward pass through the fusion model.
        logits = model(imgs, landmarks)
        loss = criterion(logits, labels)

        # Backpropagation and parameter update.
        loss.backward()
        optimizer.step()

        # Accumulate loss and accuracy statistics for this epoch.
        total_loss += loss.item() * imgs.size(0)
        
        # The predicted class is the one with the highest logit score.
        preds = logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    return total_loss / total, correct / total


def evaluate(model, loader):
    model.eval()

    total_loss = 0
    correct = 0
    total = 0

    # Disable gradient tracking during validation to save memory and compute.
    with torch.no_grad():
        for imgs, landmarks, labels in tqdm(loader, desc="Validation", leave=False):
            imgs = imgs.to(device)
            landmarks = landmarks.to(device)
            labels = labels.to(device)

            logits = model(imgs, landmarks)
            loss = criterion(logits, labels)

            total_loss += loss.item() * imgs.size(0)

            preds = logits.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    return total_loss / total, correct / total

## Train and Save the Fusion Model
Run the multimodal training loop, monitor validation accuracy, and keep the best checkpoint.

In [15]:
import torch.optim as optim
from pathlib import Path
import sys
import platform
from tqdm.notebook import tqdm

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

# === 設定模型儲存路徑 (支援跨平台防呆) ===
if 'google.colab' in sys.modules:
    # Colab 環境：直接存到 Google Drive 裡面，避免資源重置後模型遺失
    save_dir = Path("/content/drive/MyDrive/HandGesture/models")
elif platform.system() == "Windows":
    # Windows 環境：預設存到 D 槽，如果找不到路徑隊友可自行修改這邊
    save_dir = Path("D:/Hand_Gesture/model")
else:
    # Mac / Linux 本機環境：直接存於專案根目錄下的 models 資料夾
    save_dir = Path("./models")

save_dir.mkdir(parents=True, exist_ok=True)
save_path = save_dir / "fusion_mobilenetv3_landmark_aug14kdetect_best.pth"

best_val_acc = 0.0
num_epochs = 15

for epoch in range(num_epochs):
    print(f"\nEpoch {epoch + 1}/{num_epochs}")

    train_loss, train_acc = train_one_epoch(model, train_loader)
    val_loss, val_acc = evaluate(model, val_loader)

    print(
        f"train_loss={train_loss:.4f}, train_acc={train_acc:.4f}, "
        f"val_loss={val_loss:.4f}, val_acc={val_acc:.4f}"
    )

    # Keep only the checkpoint with the highest validation accuracy.
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), save_path)
        print(f"Saved best model: val_acc={best_val_acc:.4f}")

print("\nBest validation accuracy:", best_val_acc)
print("Best model saved to:", save_path)


Epoch 1/15


Training:   0%|          | 0/3083 [00:00<?, ?it/s]

Validation:   0%|          | 0/661 [00:00<?, ?it/s]

train_loss=0.1254, train_acc=0.9631, val_loss=0.0203, val_acc=0.9938
Saved best model: val_acc=0.9938

Epoch 2/15


Training:   0%|          | 0/3083 [00:00<?, ?it/s]

...

Validation:   0%|          | 0/661 [00:00<?, ?it/s]

train_loss=0.0064, train_acc=0.9981, val_loss=0.0128, val_acc=0.9971

Epoch 15/15


Training:   0%|          | 0/3083 [00:00<?, ?it/s]

Validation:   0%|          | 0/661 [00:00<?, ?it/s]

train_loss=0.0056, train_acc=0.9982, val_loss=0.0149, val_acc=0.9970

Best validation accuracy: 0.9972092143228797
Best model saved to: D:\Hand_Gesture\model\fusion_mobilenetv3_landmark_aug14kdetect_best.pth


# 4. Evaluation and Results
Evaluate the trained model on the test set and summarize model quality and size.

In [16]:
import torch
import pandas as pd
import numpy as np

class_names = ["N_A", "fist", "like", "ok", "one", "palm"]

def evaluate_detailed(model, loader, criterion=None, threshold=None):
    model.eval()

    total_loss = 0.0
    total = 0

    all_labels = []
    all_preds = []
    all_confs = []

    with torch.no_grad():
        for imgs, landmarks, labels in loader:
            imgs = imgs.to(device)
            landmarks = landmarks.to(device)
            labels = labels.to(device)

            logits = model(imgs, landmarks)

            # Optionally compute the average loss over the evaluation set.
            if criterion is not None:
                loss = criterion(logits, labels)
                total_loss += loss.item() * imgs.size(0)

            probs = torch.softmax(logits, dim=1)
            confs, preds = torch.max(probs, dim=1)

            # If a confidence threshold is provided, low-confidence predictions
            # are reassigned to the N_A class.
            if threshold is not None:
                preds = torch.where(
                    confs < threshold,
                    torch.zeros_like(preds),
                    preds
                )

            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())
            all_confs.extend(confs.cpu().numpy())

            total += labels.size(0)

    all_labels = np.array(all_labels)
    all_preds = np.array(all_preds)
    all_confs = np.array(all_confs)

    # Overall accuracy across all classes.
    accuracy = (all_labels == all_preds).mean()

    # Separate the target gesture classes from the N_A class.
    target_mask = all_labels != 0
    na_mask = all_labels == 0

    # Accuracy measured only on non-N_A classes.
    target_accuracy = (
        (all_labels[target_mask] == all_preds[target_mask]).mean()
        if target_mask.sum() > 0 else 0
    )

    # How often N_A samples are incorrectly predicted as a gesture class.
    na_false_trigger_rate = (
        (all_preds[na_mask] != 0).mean()
        if na_mask.sum() > 0 else 0
    )

    avg_loss = total_loss / total if criterion is not None else None

    metrics = {
        "loss": avg_loss,
        "accuracy": accuracy,
        "target_accuracy": target_accuracy,
        "na_false_trigger_rate": na_false_trigger_rate,
    }

    # Store per-sample prediction results for later inspection.
    result_df = pd.DataFrame({
        "true": all_labels,
        "pred": all_preds,
        "confidence": all_confs,
        "true_name": [class_names[i] for i in all_labels],
        "pred_name": [class_names[i] for i in all_preds],
    })

    cm = pd.crosstab(
        result_df["true_name"],
        result_df["pred_name"],
        rownames=["True"],
        colnames=["Pred"],
        dropna=False
    )

    # Reindex to ensure the confusion matrix always shows all 6 classes.
    cm = cm.reindex(index=class_names, columns=class_names, fill_value=0)

    return metrics, cm, result_df

## Inspect Evaluation Outputs
Display the summary metrics, confusion matrix, and a few prediction examples for manual review.

In [17]:
# Run detailed evaluation on the test set and inspect both summary metrics and examples.
metrics, cm, result_df = evaluate_detailed(
    model,
    test_loader,
    criterion=criterion,
    threshold=None
)

print(metrics)
display(cm)
display(result_df.head())

{'loss': 0.013611635780329083, 'accuracy': np.float64(0.9975403244879618), 'target_accuracy': np.float64(0.9965291955900367), 'na_false_trigger_rate': np.float64(0.0021547743643415625)}


Pred,N_A,fist,like,ok,one,palm
True,,,,,,
N_A,16208,8,9,5,5,8
fist,2,943,0,0,1,0
like,6,0,935,0,0,0
ok,0,0,0,995,0,0
one,1,0,0,0,984,0
palm,7,0,0,0,0,1024


,true,pred,confidence,true_name,pred_name
0,0,0,1.000000,N_A,N_A
1,0,0,1.000000,N_A,N_A
2,0,0,1.000000,N_A,N_A
3,0,0,0.999999,N_A,N_A
4,0,0,1.000000,N_A,N_A


## predict() with heuristic

In [7]:
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.notebook import tqdm

from inference import predict

class_names = ["N_A", "fist", "like", "ok", "one", "palm"]


def evaluate_predict_on_df(df, max_samples=None, random_state=42):
    if max_samples is not None:
        eval_df = df.sample(min(max_samples, len(df)), random_state=random_state).reset_index(drop=True)
    else:
        eval_df = df.reset_index(drop=True)

    results = []

    for row in tqdm(eval_df.itertuples(), total=len(eval_df), desc="Predict testing"):
        crop_path = Path(row.crop_path)
        landmark_path = Path(row.landmark_path)

        crop_img = np.array(Image.open(crop_path).convert("RGB"))
        landmarks = np.load(landmark_path)

        pred = predict(crop_img, landmarks)
        true = int(row.label)

        results.append({
            "true": true,
            "true_name": class_names[true],
            "pred": pred,
            "pred_name": class_names[pred],
            "correct": pred == true,
            "crop_path": str(crop_path),
            "landmark_path": str(landmark_path),
        })

    result_df = pd.DataFrame(results)

    accuracy = result_df["correct"].mean()

    target_df = result_df[result_df["true"] != 0]
    na_df = result_df[result_df["true"] == 0]

    target_accuracy = target_df["correct"].mean() if len(target_df) > 0 else 0
    na_false_trigger_rate = (na_df["pred"] != 0).mean() if len(na_df) > 0 else 0

    metrics = {
        "accuracy": accuracy,
        "target_accuracy": target_accuracy,
        "na_false_trigger_rate": na_false_trigger_rate,
        "total": len(result_df),
    }

    cm = pd.crosstab(
        result_df["true_name"],
        result_df["pred_name"],
        rownames=["True"],
        colnames=["Pred"],
        dropna=False
    )

    cm = cm.reindex(index=class_names, columns=class_names, fill_value=0)

    return metrics, cm, result_df

metrics, cm, result_df = evaluate_predict_on_df(test_df)

print(metrics)
display(cm)
display(result_df.head())

Predict testing:   0%|          | 0/21141 [00:00<?, ?it/s]

{'accuracy': np.float64(0.8742727401731233), 'target_accuracy': np.float64(0.458146182115149), 'na_false_trigger_rate': np.float64(0.00024625992735332144), 'total': 21141}


Pred,N_A,fist,like,ok,one,palm
True,,,,,,
N_A,16239,0,3,0,0,1
fist,554,392,0,0,0,0
like,92,0,849,0,0,0
ok,993,0,0,2,0,0
one,882,0,0,0,103,0
palm,133,0,0,0,0,898


,true,true_name,pred,pred_name,correct,crop_path,landmark_path
0,0,N_A,0,N_A,True,D:\Hand_Gesture\data\processed_hagrid_small_de...,D:\Hand_Gesture\data\processed_hagrid_small_de...
1,0,N_A,0,N_A,True,D:\Hand_Gesture\data\processed_hagrid_small_de...,D:\Hand_Gesture\data\processed_hagrid_small_de...
2,0,N_A,0,N_A,True,D:\Hand_Gesture\data\processed_hagrid_small_de...,D:\Hand_Gesture\data\processed_hagrid_small_de...
3,0,N_A,0,N_A,True,D:\Hand_Gesture\data\processed_hagrid_small_de...,D:\Hand_Gesture\data\processed_hagrid_small_de...
4,0,N_A,0,N_A,True,D:\Hand_Gesture\data\processed_hagrid_small_de...,D:\Hand_Gesture\data\processed_hagrid_small_de...


## Measure Checkpoint Size
Summarize the saved fusion checkpoint size and the model parameter footprint.

In [8]:
from pathlib import Path

# Report the saved checkpoint size and the number of model parameters.
file_size_mb = save_path.stat().st_size / (1024 ** 2)

print("Model file:", save_path)
print("Saved .pth size:", f"{file_size_mb:.2f} MB")

num_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

param_size_bytes = sum(p.numel() * p.element_size() for p in model.parameters())
buffer_size_bytes = sum(b.numel() * b.element_size() for b in model.buffers())

model_memory_mb = (param_size_bytes + buffer_size_bytes) / (1024 ** 2)

print("Total parameters:", f"{num_params / 1e6:.3f} M")
print("Trainable parameters:", f"{trainable_params / 1e6:.3f} M")
print("Parameter + buffer size:", f"{model_memory_mb:.2f} MB")

NameError: name 'save_path' is not defined

## Results Summary
Record the key training and evaluation outcomes for the final report.